# 🎯 Day 12 — Mock Interview Practice

**Week 2 · Day 5 · Sat Oct 24, 2026** — Difficulty: **INTERMEDIATE → ADVANCED** (8 INTERMEDIATE, 10 ADVANCED)

18 questions across 8 categories. Attempt every question on paper or a scratch cell **before** opening the solution. Time yourself: INTERMEDIATE ≈ 15 min, ADVANCED ≈ 25 min.

| # | Category | Q | Topics |
|---|---|---|---|
| 1 | SQL Advanced | Q1–Q2 | GROUPING SETS / ROLLUP subtotals · last-touch marketing attribution |
| 2 | Python | Q3–Q4 | Number of Islands (DFS) · reservoir sampling for unbounded streams |
| 3 | PySpark Coding | Q5–Q6 | recursive nested-struct flattening · snapshot diffing → SCD-1 upsert payload |
| 4 | Spark Optimization | Q7–Q8 | dynamic partition pruning + runtime filters · executor sizing & DRA |
| 5 | ETL | Q9–Q11 | VARIANT schema-on-read · SDP expectations & quarantine · hot/cold dual-path writes |
| 6 | Data Modeling | Q12–Q13 | grain declaration & degenerate dimensions · One Big Table vs star schema |
| 7 | Snowflake | Q14–Q15 | the three caching layers · Snowflake vs Databricks lakehouse |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 | failure notifications & triage runbook · listener instrumentation · self-healing pipelines |

All topics are fresh relative to days 01–11 — no repeats.

## Q1 — SQL Advanced — Subtotals in One Pass: GROUPING SETS `[INTERMEDIATE]`

**Scenario:** Finance needs the regional sales dashboard to render **detail rows, per-region subtotals, and the grand total from a single result set** — no `UNION ALL` of three queries, no client-side post-processing.

Sample data (`sales`):

| region | category | amount |
|---|---|---|
| APAC | Electronics | 1000 |
| APAC | Fashion | 600 |
| EU | Electronics | 1500 |
| EU | Fashion | 900 |
| NA | Electronics | 2500 |
| NA | Fashion | 1400 |

**Requirements**

* One row per (region, category) — the detail
* One row per region across all categories — region subtotal
* One grand-total row
* `g_region` / `g_category` marker columns so the BI layer can tell subtotal rows from detail rows

Write it with `GROUP BY GROUPING SETS` in Databricks SQL, then show the equivalent with `ROLLUP`.

**Interview follow-ups to answer out loud**

* What exactly does `GROUPING(col)` return, and why can't you identify subtotal rows with `IS NULL`?
* When is `CUBE` the wrong choice compared to explicit `GROUPING SETS`? (hint: combinations you'll never display)

In [0]:
# Solution Q1 — GROUPING SETS: detail + subtotals + grand total in one pass
from pyspark.sql import functions as F, types as T

sales = spark.createDataFrame([
    ("APAC", "Electronics", 1000.0),
    ("APAC", "Fashion", 600.0),
    ("EU", "Electronics", 1500.0),
    ("EU", "Fashion", 900.0),
    ("NA", "Electronics", 2500.0),
    ("NA", "Fashion", 1400.0),
], "region string, category string, amount double")
sales.createOrReplaceTempView("sales")

res = spark.sql("""
SELECT
  COALESCE(region, 'ALL REGIONS')      AS region,        -- NULL marks 'not grouped on'
  COALESCE(category, 'ALL CATEGORIES') AS category,
  ROUND(SUM(amount), 2)                AS total,
  GROUPING(region)   AS g_region,    -- 1 = this column was NOT in the grouping set
  GROUPING(category) AS g_category
FROM sales
GROUP BY GROUPING SETS ((region, category), (region), ())
""")
res.orderBy("g_region", "region", "g_category", "category").show(truncate=False)

# --- self-check -----------------------------------------------------------
grand          = res.filter("g_region = 1 AND g_category = 1").first()["total"]
region_totals  = {r.region: r.total for r in res.filter("g_category = 1 AND g_region = 0").collect()}
detail_total    = res.filter("g_region = 0 AND g_category = 0").agg(F.sum("total")).first()[0]
assert float(grand) == 7900.0                       # grand total row
assert float(region_totals["NA"]) == 3900.0         # region subtotal
assert abs(float(detail_total) - 7900.0) < 1e-9      # details sum to the same total
print("Q1 assertions passed")

# ROLLUP is the shorthand when the grouping sets form a hierarchy (region > category):
spark.sql("""
SELECT COALESCE(region,'ALL') region, COALESCE(category,'ALL') category, SUM(amount) total
FROM sales GROUP BY ROLLUP (region, category) ORDER BY region, category
""").show(truncate=False)

# TALKING POINTS:
# * GROUPING(col) = 1 when the row is a subtotal over that column. Never test IS NULL:
#   real NULL data in the column would look identical to a subtotal marker.
# * CUBE generates ALL 2^n combinations of the grouping columns (incl. category-only
#   totals nobody asked for). GROUPING SETS lists exactly the ones the dashboard needs.
# * One scan + one shuffle serves every level — UNION ALL of three GROUP BYs scans
#   the table three times.

## Q2 — SQL Advanced — Last-Touch Marketing Attribution `[ADVANCED]`

**Scenario (e-commerce analytics):** Marketing spends across channels (`social`, `email`, `display`, ...). Every user interaction is logged in `touches(user_id, touch_ts, channel)`; conversions live in `orders(order_id, user_id, order_ts, revenue)`. **Which channel gets credit for each order — and how much revenue per channel in total?**

Rules (classic **last-touch** model):

* For each order, credit the **latest touch by the same user at or before the order timestamp**
* Only touches within a **30-day lookback window** count
* Orders with no qualifying touch attribute to `direct`
* A touch **after** the order must never receive credit

**Requirements**

* One row per order: `order_id, revenue, attributed_channel`, then aggregate revenue by channel
* Use a **window function** to pick the winning touch per order

**Interview follow-ups**

* How does first-touch attribution change the query? Position-based 40/20/40 multi-touch?
* Why does this join blow up at scale (fan-out per user), and what's the fix? (range-join optimization, pre-aggregating one best touch per user per window)

In [0]:
# Solution Q2 — last-touch attribution: range join + ROW_NUMBER window
from pyspark.sql import functions as F, types as T

touches = spark.createDataFrame([
    ("u1", "2026-10-20 10:00:00", "social"),
    ("u1", "2026-10-20 10:30:00", "email"),   # <- u1's winning touch (10:00 < ts <= 11:00)
    ("u2", "2026-10-20 12:00:00", "social"),
    ("u4", "2026-10-20 14:30:00", "display"),  # AFTER u4's order -> must NOT get credit
], "user_id string, touch_ts string, channel string").withColumn("touch_ts", F.to_timestamp("touch_ts"))

orders = spark.createDataFrame([
    ("u1", "o100", "2026-10-20 11:00:00", 120.0),
    ("u2", "o101", "2026-10-20 12:30:00", 80.0),
    ("u3", "o102", "2026-10-20 13:00:00", 60.0),   # no touch at all -> direct
    ("u4", "o103", "2026-10-20 14:00:00", 40.0),   # only touch is AFTER the order -> direct
], "user_id string, order_id string, order_ts string, revenue double").withColumn("order_ts", F.to_timestamp("order_ts"))

touches.createOrReplaceTempView("touches")
orders.createOrReplaceTempView("orders")

attr = spark.sql("""
WITH candidates AS (
    -- every (order, touch) pair where the touch could have influenced the order:
    -- same user, touch at-or-before the order, inside the 30-day lookback
    SELECT o.order_id, o.revenue, t.channel, t.touch_ts,
           ROW_NUMBER() OVER (PARTITION BY o.order_id ORDER BY t.touch_ts DESC) AS rn
    FROM orders o
    LEFT JOIN touches t
      ON  t.user_id = o.user_id
      AND t.touch_ts <= o.order_ts
      AND t.touch_ts >= o.order_ts - INTERVAL 30 DAYS
)
SELECT order_id, revenue,
       COALESCE(channel, 'direct') AS attributed_channel   -- no candidate -> 'direct'
FROM candidates
WHERE rn = 1
""")
attr.show()

by_channel = attr.groupBy("attributed_channel").agg(F.sum("revenue").alias("revenue")).orderBy("attributed_channel")
by_channel.show()

rows = {r.attributed_channel: float(r.revenue) for r in by_channel.collect()}
assert rows == {"direct": 100.0, "email": 120.0, "social": 80.0}, rows
print("Q2 assertions passed:", rows)

# TALKING POINTS:
# * LEFT JOIN (not INNER) so untouched orders survive to become 'direct'.
# * ROW_NUMBER(ORDER BY touch_ts DESC) + rn=1 is the general 'pick one per group'
#   pattern; first-touch = ORDER BY touch_ts ASC.
# * Fan-out risk: a user with 500 touches x 20 orders = 10k candidate rows in the
#   window stage. At scale: pre-reduce touches to the last touch per user per day, or
#   use a range-join-optimized engine plan (Databricks does this for point intervals).
# * Multi-touch (position-based) = credit = CASE rn WHEN first THEN .4 WHEN last
#   THEN .4 ELSE .2 END over the same candidates — then SUM(revenue * credit).

## Q3 — Python — Number of Islands `[INTERMEDIATE]`

Classic grid-graph traversal (asked at Amazon, Google, Uber — often flavored as "count lakes in satellite imagery").

Given an `m x n` grid of `'1'`s (land) and `'0'`s (water), **count the islands** — groups of land connected **4-directionally** (horizontal/vertical only, not diagonal).

Examples:

```text
grid1 = [[1,1,1,1,0],      grid2 = [[1,1,0,0,0],
         [1,1,0,1,0],               [1,1,0,0,0],
         [1,1,0,0,0],               [0,0,1,0,0],
         [0,0,0,0,0]]               [0,0,0,1,1]]
-> 1 island                        -> 3 islands
```

**Requirements**

* `num_islands(grid)` returns the count
* Do not mutate the input grid
* O(m·n) time, O(m·n) worst-case space

**Interview follow-ups**

* DFS vs BFS here — which avoids Python's recursion limit on a 10,000×10,000 all-land grid?
* Variants: max island area, count **closed** islands (fully surrounded by water), 8-directional connectivity.

In [0]:
# Solution Q3 — flood fill with an ITERATIVE DFS (explicit stack, no recursion limit)
def num_islands(grid):
    """grid: list of list of '1'/'0'. Returns the number of 4-connected land components.
    O(m*n) time (each cell visited once), O(m*n) worst-case space. Input NOT mutated."""
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    visited = [[False] * cols for _ in range(rows)]   # separate visited matrix
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and not visited[r][c]:
                count += 1                            # found a new island
                stack = [(r, c)]                      # iterative DFS flood fill
                visited[r][c] = True
                while stack:
                    cr, cc = stack.pop()
                    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):  # 4-directional
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < rows and 0 <= nc < cols \
                           and grid[nr][nc] == "1" and not visited[nr][nc]:
                            visited[nr][nc] = True
                            stack.append((nr, nc))
    return count

g1 = [["1","1","1","1","0"],["1","1","0","1","0"],["1","1","0","0","0"],["0","0","0","0","0"]]
g2 = [["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]]
g3 = [["0"]]
g4 = [["1","0","1","0","1"]]   # diagonals do NOT connect -> 3 islands
assert num_islands(g1) == 1
assert num_islands(g2) == 3
assert num_islands(g3) == 0
assert num_islands(g4) == 3
print("Q3 assertions passed (num_islands)")

# TALKING POINTS:
# * This is connected-components on an implicit graph: nodes = land cells, edges =
#   4-adjacency. Count = number of DFS/BFS launches needed to visit all land.
# * Recursive DFS is elegant but dies with 'maximum recursion depth exceeded' on huge
#   all-land grids (one component can be 10^8 cells deep). The explicit stack keeps
#   the SAME O(m*n) bound with no interpreter limit — say this in the interview.
# * BFS with a deque is equally valid; DFS with a stack usually wins slightly on
#   constant factors because list.pop() is O(1) and the working set is smaller.
# * Variants: max-area = track len of visited cells per launch; closed islands =
#   any component touching the border is NOT closed; 8-directional = add diagonals
#   to the (dr, dc) list.

## Q4 — Python — Reservoir Sampling from an Unbounded Stream `[ADVANCED]`

**Scenario (data-engineering staple):** Maintain a uniform random sample of `k` items from a stream of **unknown, unbounded length** (e.g., sample live Kafka events for a data-quality dashboard). You see each item **exactly once**, memory is O(k), and you can't know the total count in advance.

**Requirements**

* Class `ReservoirSampler(k)` with `add(item)`
* Every item must end up in the sample with probability exactly `k / n` (uniform)
* O(1) amortized time per item, O(k) memory
* Be able to **argue the uniformity** (Algorithm R invariant)
* Verify empirically: many trials, inclusion frequencies inside a statistical band around `k/n`

**Interview follow-ups**

* Weighted reservoir sampling (A-Res / exponential keys) — one sentence each
* How to sample in Spark, and why `df.sample(fraction)` is NOT reservoir sampling
* Why does "keep each item with fixed probability p" fail to keep exactly k items as the stream grows?

In [0]:
# Solution Q4 — Algorithm R: uniform reservoir of k items from an unbounded stream
import random

class ReservoirSampler:
    """Uniform sample of k items from a stream of unknown length,
    O(1) amortized per item, O(k) memory."""
    def __init__(self, k, seed=42):
        self.k = k
        self.n = 0
        self.sample = []
        self._rng = random.Random(seed)

    def add(self, item):
        self.n += 1
        if len(self.sample) < self.k:      # fill phase: first k items go straight in
            self.sample.append(item)
        else:
            j = self._rng.randrange(self.n)  # uniform j in [0, n-1]
            if j < self.k:                   # P(replace a slot) = k/n
                self.sample[j] = item

# WHY IT'S UNIFORM (the interview proof, by induction on n):
# Invariant: after n items, every item seen so far is in the reservoir with P = k/n.
# * Item n survives: it entered with P k/n and is never evicted after arrival.
# * Item i < n survived to n-1 (P k/(n-1) by induction); item n's arrival replaces a
#   specific slot with P 1/n, so item i is evicted only if its slot is hit AND item n
#   lands in it: P(k/(n-1)) * (1 - (k/n)*(1/k)) = k/n. QED.

# EMPIRICAL CHECK: inclusion frequency of every element should be ~ k/n = 5%
N, K, TRIALS = 200, 10, 1500
counts = [0] * N
for t in range(TRIALS):
    s = ReservoirSampler(K, seed=t)
    for i in range(N):
        s.add(i)
    for i in s.sample:
        counts[i] += 1
expected = TRIALS * K / N                          # = 75
sd = (TRIALS * (K / N) * (1 - K / N)) ** 0.5       # Binomial sd
worst = max(abs(c - expected) for c in counts)
print(f"expected ~{expected:.1f} (sd {sd:.1f}), worst deviation {worst:.1f}")
assert worst < 4 * sd, worst                       # generous 4-sigma band
print("Q4 assertions passed")

s = ReservoirSampler(5, seed=7)
for i in range(1000):
    s.add(i)
print("example reservoir of 5 from a 1000-item stream:", s.sample)
assert len(s.sample) == 5 and all(0 <= i < 1000 for i in s.sample)

# TALKING POINTS:
# * 'Keep with probability p' gives a sample of ~p*n items that GROWS with n — you
#   can't bound memory. The reservoir keeps exactly k forever.
# * Weighted variant (A-Res): draw key = -u_i^(1/w_i) per item, keep the k largest
#   keys — inclusion P proportional to weight, still single pass.
# * Spark's df.sample(fraction) is Bernoulli per-row, size varies, and is NOT for
#   bounded-size streams; df.sampleBy is stratified. For exact-size uniform samples
#   use takeSample or a reservoir per partition.
# * Used in prod for: sampled audit datasets, approx distinct sketches feeding
#   dashboards, and debugging 'show me 10 random bad rows' from infinite streams.

## Q5 — PySpark — Flattening Deeply Nested Struct Columns `[INTERMEDIATE]`

**Scenario:** A vendor feed lands as `id, profile` where `profile` is a struct nested three levels deep (name, address with geo, score). Downstream consumers — and your Delta table contract — want **flat top-level columns**: `profile_name_first`, `profile_address_city`, `profile_address_geo_lat`, ...

**Requirements**

* `flatten_structs(df)` — works at **any nesting depth**, loops until no struct column remains
* Naming: `parent_child` (and `parent_child_grandchild`, ...) joined with `_`
* Do **not** explode arrays — arrays stay as-is; only structs are promoted
* Preserve non-struct columns and their order

**Interview follow-ups**

* Why not hand-write `df.select("profile.name.first", ...)`? (you must know every path up front; recursion generalizes to vendor drift)
* What happens when the vendor adds a new nested field next week?
* When is a nested struct GOOD to keep rather than flatten? (fields always read together, Parquet nested-group pruning)

In [0]:
# Solution Q5 — iterative flattening: promote struct children until none remain
from pyspark.sql import functions as F, types as T

raw = spark.createDataFrame([
    (1, {"name": {"first": "Asha", "last": "Nair"},
         "address": {"city": "Pune", "geo": {"lat": 18.5, "lon": 73.8}},
         "score": 91.5}),
    (2, {"name": {"first": "Rahul", "last": "Verma"},
         "address": {"city": "Delhi", "geo": {"lat": 28.6, "lon": 77.2}},
         "score": 84.0}),
], "id int, profile struct<name:struct<first:string,last:string>,"
   "address:struct<city:string,geo:struct<lat:double,lon:double>>,score:double>")

def flatten_structs(df, sep="_"):
    """Recursively promote every nested struct column to top-level parent_child columns.
    Arrays are left untouched (only structs are expanded)."""
    while True:
        struct_fields = [fld for fld in df.schema.fields if isinstance(fld.dataType, T.StructType)]
        if not struct_fields:              # no struct columns left -> done
            return df
        select_cols = []
        for fld in df.schema.fields:
            if isinstance(fld.dataType, T.StructType):
                for sub in fld.dataType.fields:
                    select_cols.append(
                        F.col(f"`{fld.name}`.`{sub.name}`").alias(f"{fld.name}{sep}{sub.name}"))
            else:
                select_cols.append(F.col(f"`{fld.name}`"))
        df = df.select(*select_cols)        # one level per iteration
    # each pass peels ONE level; 3-deep nesting = 3 iterations, fully schema-driven

flat = flatten_structs(raw)
flat.printSchema()
flat.show(truncate=False)

assert sorted(flat.columns) == sorted([
    "id", "profile_name_first", "profile_name_last", "profile_address_city",
    "profile_address_geo_lat", "profile_address_geo_lon", "profile_score"]), flat.columns
assert flat.count() == 2
print("Q5 assertions passed (flatten_structs)")

# TALKING POINTS:
# * The loop terminates because every select() strictly reduces total nesting depth.
# * Vendor adds profile.address.country next week -> the NEXT run just produces one
#   more column; no code change. That's schema-driven vs path-driven.
# * Guard against column-name collisions (two paths flattening to the same name):
#   in production, dedupe aliases or fail loudly on collision — silent overwrite is
#   the bug you'll be asked about.
# * Keep structs when fields are always read together (row width shrinks, nested
#   Parquet groups still prune at read). Flatten when SQL/BI consumers need columns.

## Q6 — PySpark — Snapshot Diffing: Deriving an SCD-1 Upsert Payload `[ADVANCED]`

**Scenario:** The CRM can only provide **full nightly snapshots** of customers — no CDC logs. Yesterday's snapshot and today's snapshot both arrive with identical schema `(cust_id, email, plan)`. Build the **change payload** to MERGE into `dim_customers` (SCD Type 1).

Setup:

* Old snapshot: customers 1-4. New snapshot: customers 1, 2, 4, 5
* Customer 1: plan changed (free → pro). Customer 4: email changed. Customer 2: identical. Customer 3: missing today (deleted at source). Customer 5: brand new.

**Requirements**

* Classify every customer as `NEW` / `UPDATED` / `UNCHANGED` / `DELETED`
* **Null-safe** field comparison — use `eqNullSafe` (`<=>`), because NULL vs NULL is "equal" but `=` returns NULL
* Produce an `upserts` DataFrame (NEW + UPDATED rows with new values) and a `deletes` DataFrame
* Show the Delta MERGE statements that consume them

**Interview follow-ups**

* Wide tables (100+ columns): why compare a single row hash instead of column-by-column? (`xxhash64(concat_ws(...))` — mind NULL ordering)
* Why is ONE full outer join the right shape vs a LEFT-then-RIGHT pair? (single shuffle, symmetric)
* Snapshot diffing vs log-based CDC: lag, deletes visibility, cost — when would you pick each?

In [0]:
# Solution Q6 — full outer join + null-safe comparison -> change classification
from pyspark.sql import functions as F, types as T

old_snap = spark.createDataFrame([
    (1, "asha@x.com", "free"),
    (2, "rahul@x.com", "pro"),
    (3, "meena@x.com", "pro"),
    (4, "vikram@x.com", "free"),
], "cust_id int, email string, plan string")

new_snap = spark.createDataFrame([
    (1, "asha@x.com", "pro"),      # plan changed  -> UPDATED
    (2, "rahul@x.com", "pro"),     # identical     -> UNCHANGED
    (4, "vikram2@x.com", "free"),  # email changed -> UPDATED
    (5, "kiran@x.com", "pro"),     # not in old    -> NEW
], "cust_id int, email string, plan string")
# cust 3 exists only in old -> DELETED

o = old_snap.alias("o")
n = new_snap.alias("n")
joined = o.join(n, "cust_id", "full_outer")   # single symmetric shuffle

diff = joined.select(
    F.coalesce("n.cust_id", "o.cust_id").alias("cust_id"),
    F.when(n["cust_id"].isNotNull() & o["cust_id"].isNull(), "NEW")
     .when(o["cust_id"].isNotNull() & n["cust_id"].isNull(), "DELETED")
     .when(~(o["email"].eqNullSafe(n["email"])) | ~(o["plan"].eqNullSafe(n["plan"])), "UPDATED")
     .otherwise("UNCHANGED").alias("change_type"),
    n["email"].alias("new_email"),
    n["plan"].alias("new_plan"),
)
diff.orderBy("cust_id").show()

upserts = (diff.filter("change_type IN ('NEW','UPDATED')")
              .select(F.col("cust_id"),
                      F.col("new_email").alias("email"),
                      F.col("new_plan").alias("plan")))
deletes = diff.filter("change_type = 'DELETED'").select("cust_id")
upserts.orderBy("cust_id").show()
deletes.show()

changes = {r.cust_id: r.change_type for r in diff.collect()}
assert changes == {1: "UPDATED", 2: "UNCHANGED", 3: "DELETED", 4: "UPDATED", 5: "NEW"}, changes
assert upserts.count() == 3 and deletes.count() == 1
assert diff.filter("change_type = 'UNCHANGED'").count() == 1
print("Q6 assertions passed (snapshot diff)")

# How the payload feeds Delta (SCD-1):
print("""
MERGE INTO dim_customers t
USING upserts s
  ON t.cust_id = s.cust_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO dim_customers t
USING deletes d
  ON t.cust_id = d.cust_id
WHEN MATCHED THEN DELETE;
""")

# TALKING POINTS:
# * eqNullSafe (<=>): NULL <=> NULL is TRUE. With plain = , a NULL-vs-NULL field
#   comparison silently classifies rows as UPDATED (or NULL) — a classic prod bug.
# * Wide tables: precompute row_hash = xxhash64(concat_ws('||', col1, col2, ...))
#   on both snapshots and compare TWO hash columns — 100 comparisons become 1.
#   Caveat: concat_ws drops NULLs, so coalesce every column first (or hash structs).
# * Full outer join = one shuffle for all four cases. LEFT + RIGHT anti-joins cost
#   two shuffles and still miss the UPDATED-vs-UNCHANGED split.
# * Snapshot diffing: cheap to build, but deletes only visible next snapshot, and
#   you diff EVERYTHING daily. Log CDC: exact changes + timestamps, but needs the
#   log and stateful handling. Choose by source capability, not fashion.

## Q7 — Spark Optimization — Dynamic Partition Pruning & Runtime Filters `[INTERMEDIATE]`

**Scenario:** A star-schema query filters on the **dimension**, not the fact:

```sql
SELECT f.* FROM fact_events f
JOIN dim_date d ON f.event_date = d.date_id
WHERE d.fiscal_quarter = 'FY26-Q3'
```

The fact table is petabyte-scale and partitioned by `event_date`. At **plan time** Spark cannot know which dates survive the dimension filter — so a naive engine scans everything.

**Requirements — explain and sketch:**

1. What Dynamic Partition Pruning (DPP) does at **runtime**, and the three conditions for it to engage
2. How you would **verify** DPP fired (physical plan / query profile signatures)
3. Runtime **bloom filters**: how they differ from DPP and from Delta's min/max data-skipping stats
4. What you'd check when DPP does **not** fire (join keys vs partition columns, broadcastability, subquery size)

Config family to know: `spark.sql.optimizer.dynamicPartitionPruning.*`

In [0]:
# Solution Q7 — Dynamic Partition Pruning (DPP) + runtime bloom filters
print("""
========== Solution Q7 — Dynamic Partition Pruning (DPP) ==========
PROBLEM: In a star join the selective filter sits on the DIMENSION table:
    SELECT f.* FROM fact_events f JOIN dim_date d ON f.event_date = d.date_id
    WHERE d.fiscal_quarter = 'FY26-Q3'
At PLAN time Spark cannot prune fact partitions — the needed dates are only knowable
AFTER scanning/filtering the dimension. Without help, the fact side reads EVERYTHING.

HOW DPP WORKS:
1. Spark runs the dimension filter first and (because the join is broadcastable) reuses
   the broadcast result as a partition-index lookup on the fact side.
2. At runtime, the fact scan receives a dynamic IN-list / subquery predicate restricted
   to exactly the partitions whose join keys survived on the dimension side.
3. EXPLAIN() now shows, on the fact scan:
       PartitionFilters: [isnotnull(event_date), dynamicpruningexpression(event_date IN dynamicpruning#xx))]
   (also visible as a SubqueryBroadcast node under the fact scan).

REQUIREMENTS for DPP to kick in:
* equi-join between dimension key and the fact PARTITION column (or a clustered column)
* dimension side small enough to broadcast, or a subquery below the pruning threshold
* enabled: spark.sql.optimizer.dynamicPartitionPruning.enabled = true (default)

RUNTIME BLOOM FILTERS (the join-independent cousin):
* Delta stats (min/max per column per file) give data skipping for literal filters.
* A runtime bloom filter builds a small bloom structure from the PROBE-side (dimension)
  keys and pushes it into the BUILD-side (fact) file scan — files whose stats fail the
  bloom check are skipped entirely. On Databricks this is applied automatically for
  selective joins on non-partition columns (watch for 'BloomFilter' / file-level
  'runtime filter' nodes in the query profile).

TUNING / VERIFICATION CHECKLIST:
* Confirm partition columns match the join keys — DPP cannot help otherwise.
* Check the physical plan / query profile for PartitionFilters on the fact scan.
* Keep dimension aggregates under the DPP subquery threshold
  (spark.sql.optimizer.dynamicPartitionPruning.fallbackFilterRatio governs when the
  optimizer falls back to a broadcast re-read).
""")

try:
    print("DPP enabled on this cluster:", spark.conf.get("spark.sql.optimizer.dynamicPartitionPruning.enabled"))
except Exception as e:
    print("conf not readable on serverless (managed):", type(e).__name__)
    # On serverless, DPP and runtime filtering are on by default and managed by the platform.

## Q8 — Spark Optimization — Executor Sizing & Dynamic Resource Allocation `[ADVANCED]`

**Scenario:** Your classic-compute Spark job spills to disk and GC-thrashes. You must propose a sizing and autoscaling plan to the platform team.

**Requirements — explain with numbers:**

1. Executor memory anatomy: heap (execution + storage unified), `spark.executor.memoryOverhead`, the 300MB reserve and the 0.6 usable fraction — compute usable memory for a 16GB / 4-core executor
2. Why executor heaps are usually capped at 16–32GB, and why 4–5 cores per executor is the sweet spot
3. Dynamic Resource Allocation: the knobs (`min/maxExecutors`, `executorAllocationRatio`, shuffle tracking) and when DRA actively **hurts**
4. The modern answer: how serverless / Jobs compute changes this conversation — and what YOU still tune when the platform sizes the cluster

No compute needed — produce the sizing math and the reasoning.

In [0]:
# Solution Q8 — executor sizing math + DRA knobs
print("""
========== Solution Q8 — Executor sizing & Dynamic Resource Allocation ==========
EXECUTOR MEMORY ANATOMY (per executor, classic compute):
    spark.executor.memory          -> JVM heap for execution + storage (unified)
    spark.executor.memoryOverhead  -> off-heap: netty, Python workers (>=384MB or 10% heap)
    Heap usable = (heap - 300MB reserved) * 0.6  for execution + storage
SIZING RULES OF THUMB:
* 4-5 cores per executor (keeps HDFS throughput and GC pause frequency balanced)
* Single executor heap usually 16-32GB max — bigger heaps = longer GC pauses
* skew/spill-prone jobs: fewer executors with more memory + memoryOverhead; many small
  tasks: more executors with fewer cores
WORKED EXAMPLE (executor = 4 cores, 16GB heap):
    execution+storage usable = (16GB - 0.3GB) * 0.6 ~= 9.4GB
    overhead = max(1.6GB, 384MB) = 1.6GB
    -> plan shuffle partitions so each task's spill fits comfortably inside 9.4GB / tasks_per_core

DYNAMIC RESOURCE ALLOCATION (spark.dynamicAllocation.*):
    enabled=True, minExecutors (keeps warm pool + cache), maxExecutors (cost ceiling),
    executorAllocationRatio (pack tasks densely, <1 saves cost),
    cachedExecutorIdleTimeout, shuffleTracking.enabled (removes external shuffle service dependency)
WHEN DRA HURTS:
* frequent re-executor churn on shuffle-heavy jobs (re-fetch of shuffle files)
* jobs that cache big datasets between stages (idle executors holding cache get killed)
* strict SLA bursts -> consider fixed sizing or job clusters instead
SERVERLESS / JOBS COMPUTE (the modern answer):
    Databricks serverless already autoscales elastically with no user-tuned heap at all —
    in interviews say: "on classic compute I size executors and use DRA; on serverless /
    Jobs compute the platform sizes it and I focus on shuffle & skew instead."
""")

# Worked example, computed live:
heap_gb = 16
usable = (heap_gb - 0.3) * 0.6
overhead = max(heap_gb * 0.10, 0.375)  # 10% or 384MB min
print(f"16GB executor -> usable execution+storage: {usable:.1f}GB, overhead: {overhead:.1f}GB")
print(f"total container footprint: {heap_gb + overhead:.1f}GB per executor")

## Q9 — ETL — VARIANT: Schema-on-Read for Drifting JSON `[INTERMEDIATE]`

**Scenario (2025-26 hot topic):** A partner's clickstream JSON drifts weekly — new fields appear (`referrer`), some events lack whole sections (`props`), and one producer version sent `user_id` as a **string** while everyone else sends an int. Rigid `from_json` schemas either reject rows or silently NULL out the new fields.

**Requirements — using Databricks VARIANT:**

1. Land raw payloads as a `VARIANT` column (`parse_json`) — no schema declared up front
2. **Discover** the actual schema across all rows (`schema_of_variant_agg`)
3. Extract fields **tolerantly**: type-mismatched values become NULL instead of failing the job (`try_variant_get`)
4. Show graduating to a typed struct once the schema settles (`CAST(v AS STRUCT<...>)`)

**Interview follow-ups**

* VARIANT vs `from_json` vs `get_json_object` — the cost model of each
* Where does the binary VARIANT encoding cost storage, and why is it still usually smaller than raw strings?
* When do you promote VARIANT to concrete typed columns? (column pruning + statistics)

In [0]:
# Solution Q9 — VARIANT: parse, discover, extract tolerantly, then graduate to types
from pyspark.sql import functions as F, types as T

events = spark.createDataFrame([
    ('{"event": "signup", "user_id": 7, "props": {"utm": "email", "ab_test": "B"}}',),
    ('{"event": "signup", "user_id": 8, "props": {"utm": "social", "ab_test": "A", "referrer": "app"}}',),
    ('{"event": "pageview", "user_id": 7, "url": "/pricing"}',),
    ('{"event": "signup", "user_id": "oops-not-an-int", "props": {}}',),   # type violation
], "payload string")
events.createOrReplaceTempView("events")

# 1) SCHEMA DISCOVERY across all rows — no schema declared anywhere
spark.sql("""
SELECT schema_of_variant_agg(parse_json(payload)) AS discovered_schema
FROM events
""").show(truncate=False)
# note user_id shows up as VARIANT — it holds BOTH an int and a string in the data

# 2) TOLERANT EXTRACTION — try_variant_get returns NULL on type mismatch
#    (variant_get would throw; :extraction returns VARIANT, shown with JSON quotes)
out = spark.sql("""
SELECT
  try_variant_get(v, '$.event', 'string')        AS event,
  try_variant_get(v, '$.user_id', 'int')         AS user_id,
  try_variant_get(v, '$.props.utm', 'string')   AS utm,
  try_variant_get(v, '$.url', 'string')         AS url
FROM (SELECT parse_json(payload) AS v FROM events)
""")
out.show(truncate=False)

rows = out.collect()
assert sum(1 for r in rows if r["event"] == "signup" and r["user_id"] is None) == 1, \
    "type-violating row must yield NULL, not a failed job"
assert [r for r in rows if r["event"] == "pageview"][0]["url"] == "/pricing"
assert out.filter("event = 'signup' AND utm = 'email'").count() == 1
print("Q9 assertions passed (VARIANT)")

# 3) GRADUATION: once the schema settles, cast to a typed struct (typed columns get
#    Delta statistics + column pruning + BI friendliness that raw VARIANT lacks)
spark.sql("""
SELECT CAST(parse_json(payload) AS STRUCT<event: STRING, user_id: STRING,
       props: STRUCT<utm: STRING, ab_test: STRING>>) AS typed
FROM events
""").show(truncate=False)

# TALKING POINTS:
# * get_json_object: string surgery per query, no stats, slowest at scale.
# * from_json: full reparse into a FIXED schema — any drift = NULLs or failures.
# * VARIANT: parse ONCE into a self-describing binary (sorted keys, min/max per
#   subfield) — queries prune by subfield without reparsing the string.
# * Storage: VARIANT is usually SMALLER than raw JSON strings; the cost is you keep
#   the encoded tree rather than flat columns.
# * Promote hot paths to typed columns when they need stats, joins or BI semantics.

## Q10 — ETL — Data Quality at Pipeline Birth: SDP Expectations `[ADVANCED]`

**Scenario:** Corrupt rows keep reaching the SILVER layer of your Lakeflow Spark Declarative Pipeline (SDP, formerly DLT). The team currently runs ad-hoc post-load DELETE cleanups. DQ must instead be **declared with the pipeline** and observable.

**Requirements — design + declarative code:**

1. The three severities — warn (`expect`), drop (`expect_or_drop`), fail the update (`expect_or_fail`) — and when to pick each
2. Both authoring styles: Python `@dlt.expect*` decorators and SQL `EXPECT ... ON VIOLATION DROP ROW`
3. The **quarantine-table pattern** (never silently drop in a regulated pipeline): keep bad rows in a separate table instead of discarding them
4. Observability: where pass/fail counts per expectation land (pipeline event log), and what you alert on (a rising violation **rate**, not just failures)

No compute needed — write the pipeline code as you'd commit it.

In [0]:
# Solution Q10 — row-level DQ declared WITH the pipeline (Lakeflow SDP)
print("""
========== Solution Q10 — Lakeflow SDP (Spark Declarative Pipelines) expectations ==========
ROW-LEVEL DQ DECLARED WITH THE PIPELINE, NOT IN ad-hoc SQL:

PYTHON (dlt decorators):
    import dlt

    @dlt.table(name="bronze_orders")
    def bronze_orders():
        return spark.readStream.table("kafka_raw.orders")

    @dlt.table(name="silver_orders")
    @dlt.expect("valid_amount", "amount > 0")                           # warn: keep row, count it
    @dlt.expect_or_drop("valid_status", "status IN ('paid','pending')")   # drop bad rows
    @dlt.expect_or_fail("nonzero_qty", "quantity > 0")                    # hard-fail the update
    def silver_orders():
        return dlt.read_stream("bronze_orders")

    # dict form for many rules at once:
    rules = {"valid_amount": "amount > 0", "valid_status": "status IN ('paid','pending')"}
    @dlt.table
    @dlt.expect_all_or_drop(rules)
    def silver_orders_v2(): ...

SQL (declarative, same semantics):
    CREATE OR REFRESH STREAMING TABLE silver_orders (
      EXPECT valid_amount EXPECT amount > 0 ON VIOLATION DROP ROW
    ) AS SELECT ... FROM STREAM(kafka_raw.orders);

QUARANTINE PATTERN (never silently DROP in regulated pipelines):
    @dlt.table(name="silver_orders")     # keep only good rows
    @dlt.expect_all_or_drop(rules)
    ...

    @dlt.table(name="quarantine_orders")  # keep ONLY bad rows
    def quarantine_orders():
        return dlt.read_stream("bronze_orders").where("NOT (amount > 0)")

OBSERVABILITY: every update records per-expectation pass/fail counts in the pipeline
EVENT LOG (and the pipeline UI 'Data quality' tab); query them with
    SELECT * FROM event_log(TABLE(main.my_pipeline))
    WHERE details:flow_progress.data_quality IS NOT NULL;
so you can alert on a rising violation RATE, not just hard failures.

WHEN TO USE WHICH: warn for trending drift, drop for junk you can re-source,
fail for contract violations that must page an engineer.
""")

# Key differences to state in an interview:
print("""
EXPECT SEVERITIES:
  expect           -> WARN: row kept, violation counted (drift radar)
  expect_or_drop   -> DROP: row removed from this table only (add a quarantine twin!)
  expect_or_fail   -> FAIL: the whole UPDATE fails -> pipeline stops (data contract)
WHY SDP AND NOT A POST-LOAD CLEANUP JOB?
  * rules live next to the data definition (reviewed with it, versioned with it)
  * metrics are recorded automatically per update — no bespoke audit queries
  * failure semantics are deterministic and visible in the pipeline UI
""")

## Q11 — ETL — Hot/Cold Dual-Path: Streaming + Backfill into One Delta Table `[ADVANCED]`

**Scenario:** `gold.orders` is written by BOTH a hot path (Structured Streaming `foreachBatch` MERGE, near-real-time from Kafka) and a cold path (nightly recomputation of the last 90 days from a corrected batch source). Last week the backfill died at 2 AM with `ConcurrentAppendException`.

**Requirements — explain + sketch code:**

1. Why Delta's optimistic concurrency rejects these two writers — when do writers actually conflict, and why is MERGE-on-MERGE the worst case?
2. **Pattern 1 — partition isolation**: hot path only writes the current date; backfill replaces only historical partitions (`replaceWhere`)
3. **Pattern 2 — single-writer gold**: both paths stage into silver; one consolidator MERGEs with source priority
4. **Pattern 3 — stream replay**: run backfills THROUGH the stream (new checkpoint + `trigger(availableNow=True)`)
5. Why **idempotent MERGE keys** make all three patterns safe to rerun

In [0]:
# Solution Q11 — three production patterns for streaming + backfill into one Delta table
print("""
========== Solution Q11 — Streaming + batch backfill to the SAME Delta table ==========
THE SET-UP (classic lakehouse problem):
    HOT PATH:  Structured Streaming reads Kafka, foreachBatch MERGEs near-real-time
               events into gold.orders (partitioned by event_date).
    COLD PATH: A nightly job recomputes the LAST 90 DAYS from a corrected source and
               overwrite-writes historical partitions back into the same table.
RISK: two writers -> Delta optimistic-concurrency CONFLICTS.

DELTA CONCURRENCY RULES:
* Reads never conflict. Writers conflict when their read snapshot overlaps files the
  other writer touched (MERGE-on-MERGE is the worst case: full-file rewrites).
* Typical failure: ConcurrentAppendException / METADATA_CHANGED on the backfill while
  the stream is committing every micro-batch.

PATTERN 1 — PARTITION ISOLATION (simplest, usually sufficient):
    # hot path only ever touches the CURRENT partition
    (df.withColumn("event_date", F.to_date("event_ts"))
       .write.format("delta")
       .mode("append")
       .option("replaceWhere", "event_date = current_date()")
       .save(...))          # cold path rewrites ONLY historical partitions
    * No overlap with the hot path's active-date writes -> zero conflicts.
    * Make the replaceWhere predicate EXACTLY match the partitions written
      (all-or-nothing per commit — never partial partition overwrites).

PATTERN 2 — SINGLE-WRITER GOLD (strongest):
    * Stream and batch both land in staging tables (silver_stream_stage,
      silver_backfill_stage).
    * ONE consolidator job MERGEs both into gold with source priority:
        MERGE INTO gold t
        USING (SELECT *, 'backfill' AS __source FROM backfill_stage
               UNION ALL
               SELECT *, 'stream' AS __source FROM stream_stage) s
        ON t.order_id = s.order_id AND t.event_date = s.event_date
        WHEN MATCHED AND s.__source = 'backfill' THEN UPDATE SET *
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    * Narrow the MERGE predicate (event_date + order_id) so the files read don't span
      both writers' targets.

PATTERN 3 — STREAM-ONLY (no dual writes at all):
    * Replay backfills THROUGH the stream: point the source at the backfill window
      with a NEW checkpoint location, run with trigger(availableNow=True), then restore
      the original checkpoint. The backfill is just... more stream data, and the
      idempotent MERGE (ON business key) makes replays safe.

INTERVIEW SOUND BITE: "Partition isolation for speed, single-writer for correctness,
stream replay for simplicity — and idempotent MERGE keys make all three safe to rerun."
""")

## Q12 — Data Modeling — Declaring Grain & Degenerate Dimensions `[INTERMEDIATE]`

**Scenario:** Model an e-commerce fact from the order system. Order 1001 has 3 lines; the source also carries order-level info (invoice number, shipping fee) and per-line product/store/customer references.

**Requirements — answer with design + DDL:**

1. Declare the grain of `fact_order_lines` in **one Kimball sentence**
2. Which keys become **degenerate dimensions** (stored in the fact, no dimension table) — and why `order_number` must NEVER become a dimension
3. Write the Databricks SQL DDL (with partitioning) and point out the **additive-at-grain** rule (why doesn't order-level `shipping_fee` belong here?)
4. How the degenerate keys drive the pipeline's dedup/rerun MERGE key
5. If the business later wants order **status history** — dimension retrofit or a new fact type?

In [0]:
# Solution Q12 — grain declaration + degenerate dimensions
print("""
========== Solution Q12 — Declaring grain + degenerate dimensions ==========
STEP 1 — WRITE THE GRAIN AS ONE SENTENCE (Kimball): "One row per order LINE."
    Order 1001 with 3 lines -> 3 rows. Everything else in the model derives from this.

STEP 2 — DEGENERATE DIMENSIONS: identifiers with NO descriptive attributes and extreme
    cardinality get stored IN THE FACT as bare keys — no dimension table:
      * order_number       (billions of rows, one attribute — pointless as a dim)
      * order_line_number  (also a grain component)
      * invoice_number, ticket_number, transaction_id
    They still answer "what shipped on invoice X?" without inflating the model.
    If the business LATER wants history on them (order status over time), that becomes
    a NEW fact (accumulating snapshot), never a retrofit of the dim.

DDL (Databricks SQL):
    CREATE TABLE gold.fact_order_lines (
      order_number        BIGINT,        -- degenerate dimension
      order_line_number   INT,           -- degenerate dimension (grain component)
      order_dt            DATE,
      customer_key        BIGINT,        -- FK -> dim_customer (surrogate)
      product_key         BIGINT,        -- FK -> dim_product
      store_key           BIGINT,
      quantity            INT,
      unit_price          DECIMAL(10,2),
      discount_amount     DECIMAL(10,2),
      line_amount         DECIMAL(10,2)
    )
    USING DELTA
    PARTITIONED BY (order_dt);

GRAIN DISCIPLINE CHECKS (say these in the interview):
* "Every non-key column must be additive at the declared grain" — line_amount yes;
  an order-level shipping_fee would repeat on all 3 lines and TRIPLE when summed.
  Order-level measures belong in a separate order-grain fact.
* Degenerate keys form the pipeline's idempotency/rerun key:
    MERGE INTO fact_order_lines t
    USING staged s
      ON t.order_number = s.order_number
     AND t.order_line_number = s.order_line_number
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *;
* Mixed grains in one table (line + order totals) is the #1 rookie modelling bug.
""")

## Q13 — Data Modeling — One Big Table vs Star Schema `[ADVANCED]`

**Scenario:** The ML team wants ONE wide table (events + user + product + session attributes denormalized) for feature training; BI wants the governed star schema. The platform lead asks you to settle the debate.

**Requirements — argue with trade-offs:**

1. The full comparison: query simplicity, storage, dimension-attribute update cost, column pruning, partition pruning, governance, ML reads
2. When OBT decisively **wins**, and when star **wins**
3. The hybrid medallion pattern production teams actually run in 2025-26 (star as source of truth + purpose-built derived wide tables)
4. The sound bite that ends the interview debate — what OBT really *is*

In [0]:
# Solution Q13 — One Big Table (OBT) vs star schema
print("""
========== Solution Q13 — One Big Table (OBT) vs star schema ==========
DEFINITIONS:
  star schema: facts + dims kept separate; joins at query time
  OBT: everything pre-joined into one giant wide table (all dimension
       attributes denormalized onto every fact row)

| Dimension                  | Star schema                      | One Big Table                  |
|----------------------------|----------------------------------|--------------------------------|
| Query simplicity           | joins required (BI tool handles) | zero joins, SELECT columns      |
| Storage                    | small (dims stored once)         | large (attrs repeated per row) |
| Dim attribute UPDATE       | touch dim only (1 row)           | rewrite entire fact history    |
| Column pruning / Parquet   | per-table stats                  | excellent on the wide layout   |
| Partition pruning on facts | clean                            | clean (still date-partitioned) |
| Governance / single source| dims are governed objects        | duplicated attribute copies    |
| ML feature training reads | assemble via joins              | one scan — often ~10x faster   |

WHEN OBT WINS: BI dashboards with fixed denormalized reads; ML feature tables;
  ad-hoc analytics on immutable history (event data where 'dimension changes'
  never matter).
WHEN STAR WINS: shared enterprise models; dimension attributes that CHANGE (SCD2);
  storage-sensitive lakehouses; many consumers with different join needs.

HYBRID (what Databricks teams actually do in 2025-26):
  * medallion: normalized-ish silver (schema survives upstream changes)
  * gold: STAR schema for governed BI + a small set of purpose-built WIDE tables
    for ML / top-N dashboard queries, rebuilt from silver + dims on schedule
  * the wide tables are DERIVED artifacts (documented, rebuilt, disposable),
    never sources.

INTERVIEW SOUND BITE: "OBT is a read-optimization, not a model. I keep the star as
the source of truth and materialize OBTs where scan-heavy consumers pay for
themselves."
""")

## Q14 — Snowflake — The Three Caching Layers `[INTERMEDIATE]`

**Scenario:** Your Snowflake dashboard is slow on first load each morning and instant for everyone after. Explain the full caching architecture and how to **prove** which layer is serving each query.

**Requirements — explain + verification SQL:**

1. **Result cache**: where it lives (cloud services layer — not the warehouse), what invalidates it, the 24h rolling rule, and why it survives warehouse suspend
2. **Warehouse local (SSD) cache**: micro-partition file caching, what suspend/resume does to it, multi-cluster cache pinning
3. **Remote storage** as the always-billed baseline you're optimizing away
4. The verification query — `INFORMATION_SCHEMA.QUERY_HISTORY()` and `percentage_scanned_from_cache`
5. Two tuning plays (clustering for hotter pruning; suspend-vs-rewarm economics) + how zero-copy clones interact with storage and cache

In [0]:
# Solution Q14 — Snowflake's caching layers + how to measure them
print("""
========== Solution Q14 — Snowflake's caching layers ==========
LAYER 1 — RESULT CACHE (cloud services layer, GLOBAL):
* Same normalized query text + same role + no underlying data change
  -> replays the stored result in ~ms WITHOUT a warehouse.
* Persists across warehouse suspends; invalidated when underlying micro-partitions
  change, or after 24h without reuse (rolling window).
* Cost: $0 warehouse compute (small cloud-services charge can still apply).
    SELECT ... ;  -- first run: 2.4s on warehouse
    SELECT ... ;  -- second run: 0.1s, QUERY_HISTORY shows ~0 billed warehouse ms

LAYER 2 — WAREHOUSE LOCAL CACHE (SSD on the compute nodes):
* Micro-partition COLUMN files touched by a query are cached on the warehouse's
  local SSD; repeated scans of the same data avoid remote-storage reads.
* SUSPEND clears it (nodes are released); RESUME rebuilds it lazily. On
  multi-cluster warehouses each cluster has its OWN cache — pin heavy users to
  one cluster via resource groups / separate warehouses for real cache hits.

LAYER 3 — REMOTE STORAGE (S3/Azure/GCS, the source of truth):
* Always billed on read (no cache) — this is the path you optimize away.

VERIFY & MEASURE:
    SELECT query_id, total_elapsed_time, bytes_scanned,
           percentage_scanned_from_cache
    FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
    WHERE query_text ILIKE '%fact_orders%';
    -- percentage_scanned_from_cache near 100 => L2/L1 doing the work

TUNING PLAYS:
* Cluster (recluster) or CLUSTER BY a key so hot partitions are few -> better pruning.
* Result-cache keys are sensitive: GROUP BY 1 vs GROUP BY col_name are DIFFERENT
  queries; same expression + same role + same data = same cached result.
* Suspend policy: cost of an idle warehouse vs re-warming a 300GB cache — for
  predictable daily dashboards, keeping a small warehouse warm can beat cold starts.
* Zero-copy clones share micro-partitions — cloned reads do NOT duplicate storage,
  but they DO add cache-miss reads on first touch.
""")

## Q15 — Snowflake — Snowflake vs Databricks Lakehouse `[ADVANCED]`

The 2026 interview classic: *"We're choosing between Snowflake and Databricks — make the case for each."*

**Requirements — architecture-level comparison:**

1. Architecture: Snowflake's three layers (cloud services / virtual warehouses / proprietary micro-partition storage) vs the Databricks lakehouse (open Delta in **your** storage account, Unity Catalog, decoupled compute incl. serverless + Photon)
2. Compute & performance models (vectorized C++ vs Photon + Tungsten; warehouse isolation vs job clusters)
3. Data sharing & governance (secure shares vs OPEN Delta Sharing; RBAC vs Unity Catalog with ABAC / lineage)
4. AI/ML + streaming fit
5. Cost models (credits vs DBUs — what to actually watch)
6. HONEST selection criteria — when each platform genuinely wins (and the "many enterprises run both" reality)

In [0]:
# Solution Q15 — Snowflake vs Databricks Lakehouse (the 2026 classic)
print("""
========== Solution Q15 — Snowflake vs Databricks Lakehouse ==========
ARCHITECTURE:
* Snowflake: three layers — centralized cloud services (metadata, optimizer, result
  cache), virtual warehouses (independent elastic compute, can multi-cluster),
  storage (proprietary compressed micro-partitions in object storage).
  Fully SaaS-managed; the storage format is closed.
* Databricks: LAKEHOUSE — open Delta/Parquet in YOUR cloud storage account, Unity
  Catalog governing it, decoupled compute (all-purpose/job/serverless clusters,
  Photon vectorized engine). Nothing proprietary locks the data format.

COMPUTE & PERFORMANCE:
* Snowflake: C++ vectorized execution, per-warehouse isolation, brilliant at
  concurrent BI; result cache + automatic clustering are its signature plays.
* Databricks: Photon (vectorized C++) + Tungsten codegen, Delta OPTIMIZE / liquid
  clustering; strong TPC-DS-class warehousing PLUS streaming & training workloads
  on the same data without copies.

DATA SHARING / GOVERNANCE:
* Snowflake: Secure shares + reader accounts (the consumer generally needs
  Snowflake or a marketplace listing).
* Databricks: Delta Sharing is OPEN — consumers receive Parquet/Delta without any
  Databricks footprint; Unity Catalog adds ABAC row/column policies, lineage and
  audit across catalogs including federation.

AI/ML & STREAMING:
* Snowflake: Snowpark, container services, Cortex/LLM functions — capable, but
  heavy training often still exports data out.
* Databricks: MLflow + model serving + notebooks + Structured Streaming on the SAME
  lakehouse; streaming/CDC (Lakeflow) is a first-class citizen.

COST MODEL:
* Both: consumption-priced (Snowflake credits per second per warehouse;
  Databricks DBUs per compute type) + storage. Watch: Snowflake per-byte data
  transfer and warehouse auto-suspend discipline; Databricks job/serverless compute
  for batch, snapshot/VACUUM storage growth. Openness = you can query your data
  with other engines (trino, open-source Spark) without egress-to-proprietary lock.

HONEST SELECTION CRITERIA (say this, not marketing):
* Snowflake-heavy: pure BI shop, no platform team, heavy connector/Excel users,
  sharing with Snowflake-native partners.
* Databricks-heavy: multi-workload (BI + AI + streaming + engineering), open-format
  mandate, cost pressure at scale, one platform for the whole lifecycle.
* Reality: many enterprises run BOTH and are consolidating the golden copy into
  the lakehouse.
""")

## Q16 — Monitoring & Orchestration — Failure Notifications & the 3AM Triage Runbook `[INTERMEDIATE]`

**Scenario:** You own 40 scheduled jobs. On-call engineers get paged at 3AM with no context. Design the notification + triage layer.

**Requirements — design + concrete config:**

1. What to configure once at **task level** (retries, backoff) vs **job level** (failure / staleness notifications)
2. Alert destinations (the 2025+ pattern: one Slack/PagerDuty webhook reused across many jobs) + low-sev vs high-sev routing
3. The webhook payload a triage job receives — and what that triage job does with it
4. The ordered 3AM runbook (6 steps: run timeline → task output → driver logs → upstream freshness → query profile / Spark UI → repair-run vs rollback), naming the system tables that power it
5. A standalone **freshness-alert SQL** you could attach to a SQL alert today

In [0]:
# Solution Q16 — notifications, triage job, and the ordered 3AM runbook
print("""
========== Solution Q16 — Failure notifications + 3AM triage runbook ==========
CONFIGURE ONCE (Jobs UI / API):
* Task-level: retries (2-3, with backoff) for transient errors, fixed-failure only.
* Job-level notifications: on FAILURE (and 'on staleness' for scheduled jobs) via
  e-mail + webhook. 2025+ best practice: configure an ALERT DESTINATION (Slack/Teams/
  PagerDuty webhook or email) once, attach it to many jobs.
* Route severity: low-sev -> email (daytime triage); high-sev -> PagerDuty now.

WEBHOOK PAYLOAD (what your triage job / on-call tool receives):
    { "event_type": "jobs.on_failure", "job_id": 42, "run_id": 987654,
      "task_run_id": "...", "error": "..." }
Point the webhook at a lightweight 'triage' Job (Run Job task) or a Slack workflow.

THE 3AM RUNBOOK (ordered — 90% of incidents resolved by step 3):
1. Run page -> WHICH task failed, error class, cluster/event timeline
   (system tables: system.lakeflow.job_run_timeline / job_task_run_timeline).
2. Task output (stdout) -> Python traceback / SQL error right there?
3. Cluster/driver logs (StdOut + StdErr) -> full stack trace + log4j lines around ts.
4. Upstream freshness check -> did a source table not land? (freshness SQL against
   the metrics table / DESCRIBE HISTORY max timestamp).
5. Query Profile (SQL) or Spark UI (Jobs->Stage->Task) -> OOM/spill/skew at a stage
   boundary, or a data-size explosion (row-count metric vs 7-day average).
6. Only then decide: repair-run (rerun FAILED TASKS ONLY, preserving succeeded
   outputs) vs full rerun vs rollback (Delta time-travel RESTORE).

FRESHNESS ALERT (standalone SQL — attach to a SQL alert schedule):
    SELECT IF(MAX(committed_at) < current_timestamp() - INTERVAL 90 MINUTES, 1, 0)
            AS is_stale
    FROM (SELECT MAX(commitTimestamp) AS committed_at
          FROM event_log(TABLE(gold.orders)));
""")

## Q17 — Monitoring — Custom Instrumentation with Spark Listeners `[ADVANCED]`

**Scenario:** Dashboards show the pipeline is slow, but nobody can say **why**, per batch or per query. Build instrumentation that captures the Spark engine's own metrics — with zero changes to job code.

**Requirements — code + judgment:**

1. `StreamingQueryListener` (pure PySpark): which progress events fire, and the exact fields worth persisting (inputRowsPerSecond, batchDuration, stateOperators, watermark, source metrics)
2. JVM-side `SparkListener` via `spark.extraListeners` (register without touching job code): what `onTaskEnd` / `onStageCompleted` give you (shuffle bytes, spill, attempt counts)
3. Where to write it all (a Delta metrics table) and what to alert on (p95 batchDuration vs SLA, decaying throughput)
4. What you must **NOT** record (PII, per-row logging — driver-killing patterns)
5. The one-line difference between listeners and just mining system tables / query history

In [0]:
# Solution Q17 — StreamingQueryListener + SparkListener instrumentation
print("""
========== Solution Q17 — Custom monitoring instrumentation (listeners) ==========
WHY: dashboards tell you a pipeline is slow; LISTENERS tell you WHY, per
     query/stream, with zero changes to job code.

1) STRUCTURED STREAMING — StreamingQueryListener (pure PySpark, 3.5+):
    from pyspark.sql.streaming import StreamingQueryListener

    class ProgressLogger(StreamingQueryListener):
        def on_query_started(self, event):
            ...  # record query id/name -> registry
        def on_query_progress(self, event):
            p = event.progress          # JSON-ish progress object, one per micro-batch
            record = {
                "batch_id":      p["batchId"],
                "batch_dur_ms":  p["batchDuration"],
                "input_rps":     p["inputRowsPerSecond"],
                "processed_rps": p["processedRowsPerSecond"],
                "state_rows":    p["stateOperators"][0]["numRowsTotal"],
                "watermark":     p.get("eventTime", {}).get("watermark"),
                "sources":       [s["metrics"] for s in p["sources"]],
            }
            append_to_metrics_table(record)   # async Delta append / foreachBatch
        def on_query_terminated(self, event):
            ...  # record termination + exception
    spark.streams.addListener(ProgressLogger())

2) BATCH / SQL — SparkListener (JVM side; register WITHOUT code changes):
    spark.extraListeners = com.acme.MetricsSparkListener
   Implements onTaskEnd / onStageCompleted / onJobEnd and sees exactly what the
   Spark UI sees — task metrics (shuffle read/write BYTES, spill memory, records,
   attempt count), stage duration, executor added/removed — but captured
   programmatically and exportable. For SQL-only workloads the lighter-weight
   option is QueryExecutionListener or simply mining system tables / query history.

3) WHAT TO RECORD (and what to SKIP):
   record: batchId, input/processed rates, batchDuration, source backlog estimate,
           state store size, watermark age, shuffle bytes, spill, records per stage.
   skip:   full rows, PII, per-row logging (kills the driver).

4) ALERTING ON TOP: SQL alerts over the metrics table
    SELECT batch_dur_ms FROM stream_metrics
    WHERE ts > now() - INTERVAL 1 HOUR ORDER BY ts DESC LIMIT 1;
   -> page when p95 batchDuration exceeds the SLA or processedRowsPerSecond decays.

5) LISTENERS vs SYSTEM TABLES: listeners give per-micro-batch / per-task detail for
   YOUR queries as it happens; system tables give fleet-wide AFTER-THE-FACT rollups.
   Use both: listeners for the hot path you own, system tables for everything else.
""")

## Q18 — Monitoring — Self-Healing Pipelines (Auto-Remediation) `[ADVANCED]`

**Scenario:** Leadership asks for "self-healing data pipelines." Design it so it actually works at 3AM — and **fails safely**.

**Requirements — design loop + code sketch:**

1. The loop: **DETECT → CLASSIFY → REMEDIATE → VERIFY → NOTIFY**
2. The error taxonomy that drives everything: transient infra vs resource (OOM/skew) vs data (drift/DQ) vs code — which of these is it EVER safe to auto-fix?
3. The remediation ladder: retry with backoff → **Repair Run** (rerun only failed tasks) → **rollback** (Delta time-travel RESTORE) → resize → backfill from the immutable source
4. Guardrails that make it production-safe: max attempts, circuit breaker, no destructive automated actions, an audit trail of every automated decision
5. Sketch the triage job (Jobs API `repair_run` + classification rules + audit append)

In [0]:
# Solution Q18 — self-healing loop with guardrails
print("""
========== Solution Q18 — Self-healing / auto-remediation pipelines ==========
DESIGN LOOP: DETECT -> CLASSIFY -> REMEDIATE -> VERIFY -> NOTIFY (with guardrails).

DETECT (within minutes, not morning):
* Job failure webhook -> triage job (the pattern from Q16)
* Data-level: freshness/volume/DQ metrics table checked on a schedule
* Pipeline-level: system.lakeflow.job_run_timeline for failed runs in last 24h

CLASSIFY (error taxonomy — the heart of the design):
* TRANSIENT infra (cluster start timeout, node loss, 503)    -> safe to auto-retry
* RESOURCE (OOM, spill explosion, skew straggler)            -> remediate + resize
* DATA (schema drift, DQ gate, late source)                  -> auto-quarantine, page
* CODE / logic                                                -> page only

REMEDIATE (in escalating order of strength):
1. Task-level retry with exponential backoff + jitter (idempotency required!)
2. Repair Run — rerun ONLY failed tasks (preserves succeeded outputs):
      from databricks.sdk import WorkspaceClient
      w = WorkspaceClient()
      w.jobs.repair_run(run_id=failed_run_id, rerun_all_failed_tasks=True)
3. ROLLBACK: restore the target to the last good commit before reprocessing:
      RESTORE TABLE gold.orders TO VERSION AS OF <last_good_version>;
4. RESIZE: rerun with bigger compute / more shuffle partitions (resource class)
5. BACKFILL: reprocess the affected window from the immutable source
   (idempotent MERGE on business keys = always-safe replays, as in Q11)

VERIFY + NOTIFY:
* after remediation, run the DQ gate + row-count reconciliation vs source
* every automated action is APPENDED to an audit table (who/what/when/why) —
  teammates and auditors must be able to replay every decision.

GUARDRAILS (say these or the design fails the interview):
* max 3 automated attempts per run, then page a human (no infinite retry storms)
* circuit breaker: N consecutive failures of the same task -> disable auto-retry
  until an engineer closes the incident
* blast-radius control: automated actions NEVER DROP/DELETE; restore + quarantine only
* change-freeze windows + canary (auto-remediate 1 partition first, then scale out)
""")

## 📋 Day 12 — Score Card

Attempt each question before opening the solution, then rate yourself honestly.

| Q | Topic | Level | Attempted | Solved | Confidence (1-5) | Review later |
|---|---|---|---|---|---|---|
| Q1 | GROUPING SETS / ROLLUP subtotals | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q2 | Last-touch attribution window functions | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q3 | Number of Islands (iterative DFS) | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q4 | Reservoir sampling (Algorithm R) | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q5 | Recursive struct flattening | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q6 | Snapshot diff → SCD-1 payload | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q7 | Dynamic partition pruning + runtime filters | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q8 | Executor sizing & DRA | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q9 | VARIANT schema-on-read | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q10 | SDP expectations & quarantine | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q11 | Hot/cold dual-path Delta writes | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q12 | Grain & degenerate dimensions | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q13 | OBT vs star schema | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q14 | Snowflake caching layers | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q15 | Snowflake vs Databricks lakehouse | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q16 | Notifications & triage runbook | INTERMEDIATE | ☐ | ☐ | ☐ | ☐ |
| Q17 | Listener instrumentation | ADVANCED | ☐ | ☐ | ☐ | ☐ |
| Q18 | Self-healing pipelines | ADVANCED | ☐ | ☐ | ☐ | ☐ |

**Day 12 recap:** attribution modeling and snapshot-CDC (SQL/PySpark depth), sampling theory, runtime query optimization, Delta write concurrency, VARIANT + declarative DQ (2025-26 trending), modeling debates, and the monitoring/ops triad.

**Next (Day 13, Week 2):** keep climbing the ADVANCED tier — deeper window/sequence SQL, graph algorithms in Python, Spark shuffle internals, and more production ETL scenarios.